In [ ]:
# ===== CONFIG =====
import os
INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================

In [ ]:
# repo 루트에서 실행된다고 가정하고 src/ 를 import 경로에 추가한다 (로컬 절대경로 없음)
import sys, pathlib, time

SRC = pathlib.Path.cwd() / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
import pipeline

In [ ]:
# 입력 이미지 목록
image_paths = pipeline.list_images(INPUT_DIR)
print(f"{len(image_paths)} images found in {INPUT_DIR}")

In [ ]:
# 시간 예산: 2400초 nbconvert 제한 대비 여유 120초를 남긴 2280초.
# 남은 시간/남은 장 수 기준으로 predict_one 이 회전·재시도 단계를 자동으로 줄인다 (pipeline.set_budget).
# OCR 스레드 수는 src/ocr.py 의 _THREADS(intra_op=4, inter_op=1)로 4코어 채점 서버에 고정돼 있다.
pipeline.set_budget(2280, len(image_paths))

In [ ]:
# 추론 실행. strict=False 라 이미지 한 장이 실패해도 내부에서 all-NONE 행을 만들고 계속 진행한다.
# retry_upscale=True 는 예산이 넉넉할 때만 실제로 비싼 단계까지 쓰고, 빠듯해지면 budget 이 자동으로 낮춘다.
# 진행 로그는 predict_dir 이 100장마다 찍는다.
t0 = time.time()
df = pipeline.predict_dir(INPUT_DIR, strict=False, retry_upscale=True, log_every=100)
print(f"done: {len(df)} rows in {time.time() - t0:.1f}s")

In [ ]:
# 스키마 강제 (방어적 재확인). image_id 는 문자열 그대로, year 4자리 / month·day 2자리
# zero-pad, 인식 실패는 NONE, final_date 는 셋 다 NONE 일 때만 NONE 그대로 나머지는 하이픈 결합.
def _fmt_part(v, width):
    s = "" if v is None else str(v).strip()
    if not s or s.upper() == "NONE" or not s.isdigit():
        return "NONE"
    return s.zfill(width)

def _final_date(row):
    y, m, d = row["year"], row["month"], row["day"]
    return "NONE" if (y, m, d) == ("NONE", "NONE", "NONE") else f"{y}-{m}-{d}"

df["image_id"] = df["image_id"].astype(str)
df["year"] = df["year"].apply(lambda v: _fmt_part(v, 4))
df["month"] = df["month"].apply(lambda v: _fmt_part(v, 2))
df["day"] = df["day"].apply(lambda v: _fmt_part(v, 2))
df["final_date"] = df.apply(_final_date, axis=1)
df = df[["image_id", "year", "month", "day", "final_date"]]
df.head()

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_PATH}")